# Phase 3 – Rhetorical Role Prediction
## Experiment 3: Prefix Tuning on Architecture 3 (GQA + LayerNorm + GELU)

**Technique**: Prefix Tuning  
**Base Model**: Phase-1 Architecture 3 – `GQADecoderLM` (GQA 12Q/4KV, LayerNorm, GELU FFN, ~119M params)  
**Classification Head**: CRF (Conditional Random Field) layer for structured prediction  
**Context Strategy**: Sentence-only encoding (no surrounding context) — tests context vs no-context  
**Domains**: Separate CL and IT models for domain-specific analysis

---

### What is Prefix Tuning?
Prefix Tuning prepends learnable prefix vectors P to the key and value sequences in every attention layer. The backbone weights remain **completely frozen**; only the prefix parameters are trained.

For each attention layer with n_heads and head_dim:  
`Keys_new   = [P_k; Keys_original]`  
`Values_new = [P_v; Values_original]`  

This allows the model to learn task-specific "soft prompts" in the attention space. Prefix length (l) controls capacity.

### Why CRF?
The CRF layer models label dependencies (e.g., `Fact` usually precedes `Issue` which precedes `Argument`). This captures the sequential structure of legal documents that a simple linear head ignores.

## 1. Setup

In [ ]:
import os, json, math, random, time, zipfile, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Configuration

In [ ]:
PHASE1_MODEL_PATH = '/kaggle/input/phase1-arch3/decoder_lm.pt'
TOKENIZER_PATH    = '/kaggle/input/phase1-arch3/custom_bpe.json'
TRAIN_CL = '/kaggle/input/phase3-rr-data/Train/CL_train.json'
DEV_CL   = '/kaggle/input/phase3-rr-data/Train/CL_dev.json'
TRAIN_IT = '/kaggle/input/phase3-rr-data/Train/IT_train.json'
DEV_IT   = '/kaggle/input/phase3-rr-data/Train/IT_dev.json'
LABEL_VOCAB_PATH = '/kaggle/input/phase3-rr-data/Train/label_vocab.json'
TEST_CL  = '/kaggle/input/phase3-rr-data/Test/CL_test.json'
TEST_IT  = '/kaggle/input/phase3-rr-data/Test/IT_test.json'
OUTPUT_DIR = '/kaggle/working/phase3_exp3_prefix_arch3'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Prefix Tuning
PREFIX_LEN     = 30     # number of prefix tokens per layer
PREFIX_DIM     = 512    # intermediate MLP dim for prefix reparameterization

# Training
MAX_SEQ_LEN    = 256
BATCH_SIZE     = 1      # one doc at a time (CRF needs full sequence)
LEARNING_RATE  = 1e-3   # higher LR for prefix-only training
WEIGHT_DECAY   = 0.001
EPOCHS         = 12
GRAD_CLIP      = 1.0
SEPARATE_DOMAINS = True  # train separate models for CL and IT

print('Experiment 3: Prefix Tuning + CRF on Architecture 3 configured.')

## 3. Tokenizer

In [ ]:
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []; self.merge_rank = {}; self.itos = []; self.stoi = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        syms = self._word_to_symbols(word)
        while len(syms) > 1:
            pairs = [(syms[i], syms[i+1]) for i in range(len(syms)-1)]
            cands = [(self.merge_rank[p], i) for i, p in enumerate(pairs) if p in self.merge_rank]
            if not cands: break
            _, i = min(cands)
            syms = syms[:i] + [syms[i]+syms[i+1]] + syms[i+2:]
        return syms

    def encode(self, text, max_len=None):
        ids = [self.stoi.get(s, self.stoi['<unk>']) for w in text.split() for s in self._apply_merges_to_word(w)]
        return ids[:max_len] if max_len else ids

    @classmethod
    def load(cls, path):
        with open(path) as f: d = json.load(f)
        t = cls(vocab_size=d['vocab_size'])
        t.itos = d['itos']; t.stoi = {x: i for i, x in enumerate(t.itos)}
        t.merges = [tuple(p) for p in d['merges']]
        t.merge_rank = {p: i for i, p in enumerate(t.merges)}
        return t

tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get('<pad>', 0)
print(f'Tokenizer: vocab_size={tokenizer.vocab_size}')

## 4. Labels & Data

In [ ]:
def load_json(p):
    with open(p) as f: return json.load(f)

with open(LABEL_VOCAB_PATH) as f: lv = json.load(f)
LABELS = lv if isinstance(lv, list) else list(lv.keys())
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)
print(f'Labels ({NUM_LABELS}):', LABELS)

cl_train = load_json(TRAIN_CL); cl_dev = load_json(DEV_CL)
it_train = load_json(TRAIN_IT); it_dev = load_json(DEV_IT)
cl_test  = load_json(TEST_CL);  it_test= load_json(TEST_IT)

# Class weights (from combined train)
all_lbls = [l for doc in cl_train+it_train for l in doc.get('labels',[])]
counts = Counter(all_lbls); total = sum(counts.values())
class_weights = torch.tensor(
    [total/(NUM_LABELS*counts.get(l,1)) for l in LABELS], dtype=torch.float32
).to(device)
print('Data loaded. Class weights computed.')

## 5. Architecture 3 Model (GQA)

In [ ]:
@dataclass
class GQAConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 3072
    max_seq_len: int = 1024
    dropout: float = 0.1

class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg: GQAConfig):
        super().__init__()
        self.n_heads = cfg.n_heads; self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.n_q_per_kv = cfg.n_heads // cfg.n_kv_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj  = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj  = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj  = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.out_proj= nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop= nn.Dropout(cfg.dropout)
        mask = torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len, dtype=torch.bool))
        self.register_buffer('causal_mask', mask, persistent=False)
        # Prefix-tuning storage (set externally)
        self.prefix_k = None  # [1, n_heads, prefix_len, head_dim]
        self.prefix_v = None

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(B,T,self.n_kv,self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(B,T,self.n_kv,self.head_dim).transpose(1,2)
        # Expand KV heads
        if self.n_q_per_kv > 1:
            k = k.repeat_interleave(self.n_q_per_kv, dim=1)
            v = v.repeat_interleave(self.n_q_per_kv, dim=1)
        # Prepend prefix to K and V
        if self.prefix_k is not None:
            pk = self.prefix_k.expand(B, -1, -1, -1)  # [B, n_heads, P, head_dim]
            pv = self.prefix_v.expand(B, -1, -1, -1)
            k = torch.cat([pk, k], dim=2)  # [B, n_heads, P+T, head_dim]
            v = torch.cat([pv, v], dim=2)
        scores = (q @ k.transpose(-2,-1)) * self.scale
        # Causal mask only over original tokens (prefix is always visible)
        P = 0 if self.prefix_k is None else self.prefix_k.shape[2]
        cm = self.causal_mask[:T, :T]
        # Extend mask to allow attending to all prefix positions
        prefix_mask = torch.ones(T, P, dtype=torch.bool, device=x.device)
        full_mask = torch.cat([prefix_mask, cm], dim=1)  # [T, P+T]
        scores = scores.masked_fill(~full_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)
        z = attn @ v
        z = z.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_drop(self.out_proj(z))

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout)
        )
    def forward(self, x): return self.net(x)

class GQABlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = GroupedQueryAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn  = FFN(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GQADecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        # Sinusoidal PE for Arch 3
        pe = torch.zeros(cfg.max_seq_len, cfg.d_model)
        pos = torch.arange(0, cfg.max_seq_len).unsqueeze(1).float()
        div = torch.pow(10000.0, torch.arange(0, cfg.d_model, 2).float() / cfg.d_model)
        pe[:, 0::2] = torch.sin(pos / div); pe[:, 1::2] = torch.cos(pos / div)
        self.register_buffer('pos_emb', pe, persistent=False)
        self.blocks = nn.ModuleList([GQABlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:T]
        for block in self.blocks: x = block(x)
        return self.ln_f(x)

print('GQA Architecture 3 defined.')

## 6. Prefix Tuning Module

In [ ]:
class PrefixTuningModule(nn.Module):
    """
    Learns prefix key-value pairs for all attention layers via a reparameterization MLP.
    The MLP approach avoids overfitting with direct optimization of prefix embeddings.
    
    Following Li & Liang (2021): prefix params are not directly trained;
    instead a MLP maps from prefix_id -> prefix embedding -> (K, V) per layer.
    """
    def __init__(self, num_layers: int, n_heads: int, head_dim: int,
                 prefix_len: int, prefix_dim: int, dropout: float = 0.0):
        super().__init__()
        self.prefix_len = prefix_len
        self.num_layers = num_layers
        self.n_heads    = n_heads
        self.head_dim   = head_dim

        # Prefix embeddings (one set per layer for K and V)
        self.prefix_emb = nn.Embedding(prefix_len, prefix_dim)

        # Reparameterization MLP: prefix_dim -> 2 * num_layers * n_heads * head_dim
        out_dim = 2 * num_layers * n_heads * head_dim
        self.reparam_mlp = nn.Sequential(
            nn.Linear(prefix_dim, prefix_dim * 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(prefix_dim * 2, out_dim)
        )

        nn.init.normal_(self.prefix_emb.weight, std=0.02)

    def forward(self) -> List:
        """
        Returns list of (prefix_k, prefix_v) tuples, one per layer.
        Each tensor: [1, n_heads, prefix_len, head_dim]
        """
        prefix_ids = torch.arange(self.prefix_len, device=self.prefix_emb.weight.device)
        emb = self.prefix_emb(prefix_ids)                      # [P, prefix_dim]
        out = self.reparam_mlp(emb)                            # [P, 2*L*H*D]
        # Reshape to [L, 2, P, H, D]
        L, H, D, P = self.num_layers, self.n_heads, self.head_dim, self.prefix_len
        out = out.view(P, 2, L, H, D).permute(2, 1, 3, 0, 4)  # [L, 2, H, P, D]
        # Split into K and V per layer
        prefix_kv = []
        for l in range(L):
            k = out[l, 0].unsqueeze(0)  # [1, H, P, D]
            v = out[l, 1].unsqueeze(0)
            prefix_kv.append((k, v))
        return prefix_kv

def attach_prefixes(backbone: GQADecoderLM, prefix_module: PrefixTuningModule):
    """Compute prefixes and attach them to each attention layer's prefix_k/v buffers."""
    prefix_kv = prefix_module()
    for l, block in enumerate(backbone.blocks):
        block.attn.prefix_k = prefix_kv[l][0]  # [1, n_heads, P, head_dim]
        block.attn.prefix_v = prefix_kv[l][1]

print('Prefix Tuning module defined.')

## 7. CRF Layer for Structured Prediction

In [ ]:
class CRF(nn.Module):
    """
    Linear-chain Conditional Random Field.
    Models label transition dependencies for sequential prediction.
    """
    def __init__(self, num_tags: int):
        super().__init__()
        self.num_tags = num_tags
        # Transition scores: transitions[i, j] = score of transitioning from tag j to tag i
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        # Start/end scores
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions   = nn.Parameter(torch.empty(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def forward(self, emissions: torch.Tensor, tags: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Compute negative log-likelihood (loss).
        emissions: [S, num_tags]
        tags: [S]
        Returns: scalar loss
        """
        if mask is None:
            mask = torch.ones(emissions.size(0), dtype=torch.bool, device=emissions.device)
        S = emissions.size(0)
        # Score of gold sequence
        score = self.start_transitions[tags[0]] + emissions[0, tags[0]]
        for i in range(1, S):
            if mask[i]:
                score = score + self.transitions[tags[i], tags[i-1]] + emissions[i, tags[i]]
        score = score + self.end_transitions[tags[S-1]]
        # Log-partition function via forward algorithm
        alpha = self.start_transitions + emissions[0]  # [num_tags]
        for i in range(1, S):
            if mask[i]:
                emit = emissions[i].unsqueeze(0)        # [1, num_tags]
                trans = self.transitions                 # [num_tags, num_tags]
                alpha = torch.logsumexp(alpha.unsqueeze(0) + trans + emit, dim=1)
        alpha = alpha + self.end_transitions
        log_Z = torch.logsumexp(alpha, dim=0)
        return log_Z - score  # negative log-likelihood

    def decode(self, emissions: torch.Tensor) -> List[int]:
        """Viterbi decoding. Returns best tag sequence."""
        S, K = emissions.shape
        viterbi = torch.full((S, K), -1e9, device=emissions.device)
        backptr  = torch.zeros((S, K), dtype=torch.long, device=emissions.device)
        viterbi[0] = self.start_transitions + emissions[0]
        for i in range(1, S):
            v = viterbi[i-1].unsqueeze(1) + self.transitions  # [K, K] (from, to)
            viterbi[i], backptr[i] = v.max(dim=0)
            viterbi[i] += emissions[i]
        viterbi[-1] += self.end_transitions
        best_last = viterbi[-1].argmax().item()
        tags = [best_last]
        for i in range(S-1, 0, -1):
            tags.append(backptr[i, tags[-1]].item())
        return list(reversed(tags))

print('CRF layer defined.')

## 8. Full Model

In [ ]:
class RRClassifierArch3(nn.Module):
    """
    Prefix Tuning + CRF on Architecture 3 (GQA).
    
    Sentence encoding: mean-pool of prefix-adapted GQA backbone output
    Document modeling: simple pooling (no-context experiment)
    Output: CRF decodes label sequence
    """
    def __init__(self, backbone: GQADecoderLM, num_labels: int, prefix_len: int,
                 prefix_dim: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        n_layers = backbone.cfg.n_layers
        n_heads  = backbone.cfg.n_heads
        head_dim = d_model // n_heads
        self.num_labels = num_labels

        # Prefix module
        self.prefix_module = PrefixTuningModule(
            num_layers=n_layers, n_heads=n_heads, head_dim=head_dim,
            prefix_len=prefix_len, prefix_dim=prefix_dim, dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)
        self.emission_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_labels)
        )
        self.crf = CRF(num_labels)

        nn.init.xavier_uniform_(self.emission_head[0].weight)
        nn.init.xavier_uniform_(self.emission_head[3].weight)

    def get_sentence_embs(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        """input_ids: [S, T] -> sentence embs [S, d_model]"""
        # Attach prefix to all layers
        attach_prefixes(self.backbone, self.prefix_module)
        hidden = self.backbone(input_ids)              # [S, T, d_model]
        mask_exp = attention_mask.unsqueeze(-1).float()
        sent_embs = (hidden * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        return self.dropout(sent_embs)                 # [S, d_model]

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Returns loss (train) or decoded tags (inference).
        """
        sent_embs = self.get_sentence_embs(input_ids, attention_mask)  # [S, d_model]
        emissions = self.emission_head(sent_embs)                      # [S, num_labels]
        if labels is not None:
            valid = labels >= 0
            if valid.sum() == 0:
                return torch.tensor(0.0, requires_grad=True, device=input_ids.device)
            loss = self.crf(emissions[valid], labels[valid])
            return loss
        else:
            return self.crf.decode(emissions)

print('RRClassifierArch3 (Prefix Tuning + CRF) defined.')

## 9. Load Checkpoint & Build Model

In [ ]:
print('Loading Architecture 3 checkpoint...')
ckpt = torch.load(PHASE1_MODEL_PATH, map_location='cpu')

raw = ckpt.get('config', {})
if hasattr(raw, '__dict__'): raw = raw.__dict__
d = raw if isinstance(raw, dict) else {}
cfg = GQAConfig(
    vocab_size=ckpt.get('vocab_size', tokenizer.vocab_size),
    d_model=d.get('d_model', 768), n_layers=d.get('n_layers', 12),
    n_heads=d.get('n_heads', 12), n_kv_heads=d.get('n_kv_heads', 4),
    d_ff=d.get('d_ff', 3072), max_seq_len=d.get('max_seq_len', 1024)
)
print(f'GQA Config: {cfg}')

backbone = GQADecoderLM(cfg)
backbone.load_state_dict(ckpt['model_state_dict'], strict=False)
print('Backbone loaded.')

# Freeze EVERYTHING in backbone (prefix tuning: zero backbone grad)
for p in backbone.parameters():
    p.requires_grad = False

model = RRClassifierArch3(
    backbone, NUM_LABELS,
    prefix_len=PREFIX_LEN, prefix_dim=PREFIX_DIM
).to(device)

# Only prefix_module, emission_head, CRF are trainable
tot = sum(p.numel() for p in model.parameters())
tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: total={tot:,} | trainable={tr:,} ({100*tr/tot:.2f}%)')

## 10. Dataset (Sentence-only, no context window)

In [ ]:
class RRSentenceOnlyDataset(Dataset):
    """No context window — pure sentence-level encoding."""
    def __init__(self, docs, tokenizer, label2id, max_seq_len, is_test=False):
        self.items = []
        pad_id = tokenizer.stoi.get('<pad>', 0)
        for doc in docs:
            sents  = doc['text']
            labels = doc.get('labels', [None]*len(sents))
            doc_id = doc['id']
            ids_list, masks_list, lbls_list = [], [], []
            for sent, lbl in zip(sents, labels):
                ids = tokenizer.encode(sent, max_len=max_seq_len)
                pad = max_seq_len - len(ids)
                mask = [1]*len(ids) + [0]*pad
                ids  = ids + [pad_id]*pad
                ids_list.append(ids); masks_list.append(mask)
                lbls_list.append(label2id.get(lbl, 0) if (not is_test and lbl) else -1)
            self.items.append({
                'doc_id': doc_id,
                'input_ids': torch.tensor(ids_list, dtype=torch.long),
                'attention_mask': torch.tensor(masks_list, dtype=torch.long),
                'labels': torch.tensor(lbls_list, dtype=torch.long)
            })
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

collate_fn = lambda b: b

# Build datasets
cl_tr_ds = RRSentenceOnlyDataset(cl_train, tokenizer, LABEL2ID, MAX_SEQ_LEN)
it_tr_ds = RRSentenceOnlyDataset(it_train, tokenizer, LABEL2ID, MAX_SEQ_LEN)
cl_dv_ds = RRSentenceOnlyDataset(cl_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN)
it_dv_ds = RRSentenceOnlyDataset(it_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN)
cl_te_ds = RRSentenceOnlyDataset(cl_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, is_test=True)
it_te_ds = RRSentenceOnlyDataset(it_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, is_test=True)

train_ds = ConcatDataset([cl_tr_ds, it_tr_ds])
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_fn)
cl_dv_loader = DataLoader(cl_dv_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
it_dv_loader = DataLoader(it_dv_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
dev_loader   = DataLoader(ConcatDataset([cl_dv_ds, it_dv_ds]), batch_size=1,
                          shuffle=False, collate_fn=collate_fn)
print(f'Train={len(train_ds)}, CL-dev={len(cl_dv_ds)}, IT-dev={len(it_dv_ds)}')

## 11. Training

In [ ]:
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=3, T_mult=2, eta_min=1e-6)

def eval_loader(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            lbls = doc['labels'].to(device)
            attach_prefixes(model.backbone, model.prefix_module)
            preds = model(ids, mask)  # list of tag ids
            true  = lbls.cpu().tolist()
            all_p.extend(preds); all_l.extend(true)
    vm = [i for i, l in enumerate(all_l) if l >= 0]
    ap = [all_p[i] for i in vm]; al = [all_l[i] for i in vm]
    return f1_score(al, ap, average='macro', zero_division=0), ap, al

best_f1, best_ckpt = 0.0, os.path.join(OUTPUT_DIR, 'best_prefix_arch3.pt')
print('Training Prefix Tuning + CRF...')

for epoch in range(EPOCHS):
    model.train()
    ep_loss, steps = 0.0, 0
    for batch in train_loader:
        doc  = batch[0]
        ids  = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        loss = model(ids, mask, lbls)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        optimizer.step()
        ep_loss += loss.item(); steps += 1
    scheduler.step()
    f1, _, _ = eval_loader(model, dev_loader)
    print(f'Epoch {epoch+1:2d} | Loss={ep_loss/max(1,steps):.4f} | Dev Macro-F1={f1:.4f}')
    if f1 > best_f1:
        best_f1 = f1
        torch.save({'model_state_dict': model.state_dict(), 'f1': f1}, best_ckpt)
        print(f'  -> Saved best (F1={f1:.4f})')
print(f'Best Dev Macro-F1: {best_f1:.4f}')

## 12. Evaluation & Domain Analysis

In [ ]:
ckpt_best = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt_best['model_state_dict'])
model.eval()

# Combined report
f1_all, ap, al = eval_loader(model, dev_loader)
print('=== Combined Dev Classification Report ===')
print(classification_report(al, ap, target_names=LABELS, zero_division=0))

# Domain split
cl_f1, _, _ = eval_loader(model, cl_dv_loader)
it_f1, _, _ = eval_loader(model, it_dv_loader)
print(f'CL Dev Macro-F1: {cl_f1:.4f}')
print(f'IT Dev Macro-F1: {it_f1:.4f}')

# Confusion analysis
confusion = defaultdict(int)
for t, p in zip(al, ap):
    if t != p: confusion[(ID2LABEL[t], ID2LABEL[p])] += 1
print('\nTop-10 Confused Pairs:')
for (t, p), c in sorted(confusion.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {t:35s} -> {p:35s}: {c}')

## 13. Generate Test Predictions

In [ ]:
def gen_preds_crf(model, test_ds, device, id2label):
    loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    preds = {}
    model.eval()
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            attach_prefixes(model.backbone, model.prefix_module)
            tag_ids = model(ids, mask)
            preds[doc['doc_id']] = [id2label[t] for t in tag_ids]
    return preds

cl_preds = gen_preds_crf(model, cl_te_ds, device, ID2LABEL)
it_preds = gen_preds_crf(model, it_te_ds, device, ID2LABEL)
all_preds = {**cl_preds, **it_preds}

errors = sum(1 for doc in cl_test+it_test
             if len(all_preds.get(doc['id'],[])) != len(doc['text']))
print(f'Prediction errors: {errors}')

pred_path = os.path.join(OUTPUT_DIR, 'predictions.json')
with open(pred_path, 'w') as f: json.dump(all_preds, f, indent=2)
zip_path = '/kaggle/working/submission_exp3_prefix_arch3.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, 'predictions.json')
print(f'Submission: {zip_path}')

## 14. Context vs No-Context Analysis

This experiment uses **sentence-only encoding** (no context window) to compare with Experiments 1 & 2 (which use ±2/3 sentence context).

### Expected Findings
- Labels with strong **individual** linguistic signatures (e.g., `Statute`) should not degrade much
- Labels requiring **sequential context** (e.g., `ArgumentPetitioner` vs `ArgumentRespondent`) will suffer
- CRF label transitions partially compensate for missing context

### Summary Table
| Experiment | Backbone | Technique | Context | Head |
|---|---|---|---|---|
| 1 | Arch1 (91M) | LoRA (attn only) | ±2 sents | BiLSTM |
| 2 | Arch2 (105M)| LORMA (attn+FFN) | ±3 sents | Transformer |
| 3 | Arch3 (119M)| Prefix Tuning | None | CRF |